# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mvdu12/ml-internship-starter/blob/main/work/notebooks/capstone.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [1]:
print(
    "Research question: which pages in a content portfolio should a team review "
    "first for a refresh, given that not every page needs equal attention?\n\n"
    "Decision this supports: a content team's weekly triage -- where to spend "
    "limited editorial time for the best chance of arresting a decline before it "
    "gets expensive to reverse.\n\n"
    "Unit of analysis: one content page (one row = one page, n=30,000).\n"
    "Output: a ranked queue with a probability, an action tier, and a reason "
    "code (ML-10's action_playbook_queue.csv).\n"
    "Cost of a wrong call: reviewing a page that wasn't declining wastes editor "
    "time; missing a page that WAS declining lets it keep losing visibility "
    "unnoticed. Neither error is catastrophic per-page, which is exactly why a "
    "ranked prioritization tool -- not a hard automated decision -- fits this "
    "problem."
)


Research question: which pages in a content portfolio should a team review first for a refresh, given that not every page needs equal attention?

Decision this supports: a content team's weekly triage -- where to spend limited editorial time for the best chance of arresting a decline before it gets expensive to reverse.

Unit of analysis: one content page (one row = one page, n=30,000).
Output: a ranked queue with a probability, an action tier, and a reason code (ML-10's action_playbook_queue.csv).
Cost of a wrong call: reviewing a page that wasn't declining wastes editor time; missing a page that WAS declining lets it keep losing visibility unnoticed. Neither error is catastrophic per-page, which is exactly why a ranked prioritization tool -- not a hard automated decision -- fits this problem.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [2]:
import pandas as pd
import numpy as np

url = 'https://raw.githubusercontent.com/Mvdu12/ml-internship-starter/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print(f"Release: content_refresh_anonymized.csv (starter export), {df.shape[0]:,} rows, {df.shape[1]} columns")
print("Window: trailing 90-day performance metrics; trend_direction from 30d-vs-prev-30d impressions.")
print(f"Base rate (is_declining_label): {df['is_declining_label'].mean():.3f}")
print()
print("Deliberately excluded (full reasoning in ML-05):")
print(
    "  - trend_direction / trend_pct: these DEFINE the label -- used only to audit "
    "signals and score the model, never as a feature.\n"
    "  - content_id / client_id: pseudonymous identifiers, used only for grouping "
    "and joins, never as model inputs.\n"
    "  - provider_used / model_used: flagged 'not a model feature' in the data "
    "dictionary -- production metadata, not a content signal.\n"
    "  - *_tier columns: bucketed duplicates of numeric columns already used "
    "directly.\n"
    "  - FlyRank's own product flags (health_score, priority_score, action_type): "
    "not present in this dataset; would be circular (decision-derived leakage) if "
    "they were.\n"
    "Public-safe: no client names, no raw queries, no real URLs anywhere in this "
    "repo -- content_id/client_id are pseudonymous hashes."
)


Release: content_refresh_anonymized.csv (starter export), 30,000 rows, 45 columns
Window: trailing 90-day performance metrics; trend_direction from 30d-vs-prev-30d impressions.
Base rate (is_declining_label): 0.542

Deliberately excluded (full reasoning in ML-05):
  - trend_direction / trend_pct: these DEFINE the label -- used only to audit signals and score the model, never as a feature.
  - content_id / client_id: pseudonymous identifiers, used only for grouping and joins, never as model inputs.
  - provider_used / model_used: flagged 'not a model feature' in the data dictionary -- production metadata, not a content signal.
  - *_tier columns: bucketed duplicates of numeric columns already used directly.
  - FlyRank's own product flags (health_score, priority_score, action_type): not present in this dataset; would be circular (decision-derived leakage) if they were.
Public-safe: no client names, no raw queries, no real URLs anywhere in this repo -- content_id/client_id are pseudonymo

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [3]:
print(
    "Label: is_declining_label = (trend_direction == 'down'), i.e. impressions "
    "fell more than 10% over the trailing 30 days vs the prior 30.\n\n"
    "Baseline (ML-07/ML-08): a transparent rule -- "
    "freshness_risk_percentile x visibility_percentile (days_since_last_update, "
    "impressions_90d) -- no fitting, fully readable, the number every model here "
    "has to beat.\n\n"
    "Model (ML-08): Logistic Regression, then Random Forest -- the "
    "'yes/no with an observed label' shape from the training-honest-models menu, "
    "chosen because it's directly comparable to the baseline via precision@K on "
    "the same label.\n\n"
    "Features: 16 numeric (visibility, engagement, age/freshness, keyword "
    "economics) + 3 categorical (content_type, main_intent, competition_level), "
    "with missing-value flags rather than silent fills. Full feature notes in "
    "ML-05.\n\n"
    "Validation design: GroupShuffleSplit by client_id, 80/20 -- not a random "
    "row-level split, because pages from the same client share house style and "
    "audience; a random split lets the model partly memorize 'which client is "
    "this' instead of learning decline risk (quantified in ML-09, chart below).\n\n"
    "Leakage checks (ML-05, re-run in ML-09): adding trend_pct or "
    "trend_direction as a feature collapses/inflates AUC to 1.000 -- confirms "
    "the final feature set is free of the two label-derived columns."
)


Label: is_declining_label = (trend_direction == 'down'), i.e. impressions fell more than 10% over the trailing 30 days vs the prior 30.

Baseline (ML-07/ML-08): a transparent rule -- freshness_risk_percentile x visibility_percentile (days_since_last_update, impressions_90d) -- no fitting, fully readable, the number every model here has to beat.

Model (ML-08): Logistic Regression, then Random Forest -- the 'yes/no with an observed label' shape from the training-honest-models menu, chosen because it's directly comparable to the baseline via precision@K on the same label.

Features: 16 numeric (visibility, engagement, age/freshness, keyword economics) + 3 categorical (content_type, main_intent, competition_level), with missing-value flags rather than silent fills. Full feature notes in ML-05.

Validation design: GroupShuffleSplit by client_id, 80/20 -- not a random row-level split, because pages from the same client share house style and audience; a random split lets the model partly mem

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [4]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

freshness_risk = df['days_since_last_update'].rank(method='average', pct=True)
visibility = np.log1p(df['impressions_90d']).rank(method='average', pct=True)
df['baseline_action_score'] = freshness_risk * visibility

numeric_feats = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'content_age_days', 'days_since_last_update',
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'engaged_sessions_90d',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
]
cat_feats = ['content_type', 'main_intent', 'competition_level']
X = df[numeric_feats + cat_feats].copy()
for c in numeric_feats:
    X[c + '_missing'] = X[c].isna().astype(int)
    X[c] = X[c].fillna(0)
X = pd.get_dummies(X, columns=cat_feats, dummy_na=True)
y = df['is_declining_label']
groups = df['client_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr, te = next(gss.split(X, y, groups))
Xtr, Xte = X.iloc[tr].copy(), X.iloc[te].copy()
ytr, yte = y.iloc[tr], y.iloc[te]
baseline_test = df['baseline_action_score'].iloc[te]

sc = StandardScaler()
Xtr_s, Xte_s = Xtr.copy(), Xte.copy()
Xtr_s[numeric_feats] = sc.fit_transform(Xtr[numeric_feats])
Xte_s[numeric_feats] = sc.transform(Xte[numeric_feats])
logreg = LogisticRegression(max_iter=1000, random_state=42).fit(Xtr_s, ytr)
p_lr = logreg.predict_proba(Xte_s)[:, 1]

rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1).fit(Xtr, ytr)
p_rf = rf.predict_proba(Xte)[:, 1]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

rows = []
for k in [50, 200, 500]:
    rows.append(['baseline', k, round(precision_at_k(baseline_test.values, yte.values, k), 3)])
    rows.append(['logistic_regression', k, round(precision_at_k(p_lr, yte.values, k), 3)])
    rows.append(['random_forest', k, round(precision_at_k(p_rf, yte.values, k), 3)])
comparison = pd.DataFrame(rows, columns=['method', 'k', 'precision_at_k'])
print("Model vs baseline, same grouped test split, same label:\n")
print(comparison.pivot(index='k', columns='method', values='precision_at_k'))
print(f"\nTest-set base rate: {yte.mean():.3f}  (n_test={len(Xte):,})")

# split-honesty check (ML-09)
tr_r, te_r = train_test_split(np.arange(len(X)), test_size=0.2, random_state=42, stratify=y)
rf_r = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1).fit(X.iloc[tr_r], y.iloc[tr_r])
p_rf_r = rf_r.predict_proba(X.iloc[te_r])[:, 1]
print(
    f"\nSplit honesty (ML-09): Random Forest precision@500 is "
    f"{precision_at_k(p_rf_r, y.iloc[te_r].values, 500):.3f} under a random "
    f"row-level split vs {precision_at_k(p_rf, yte.values, 500):.3f} under the "
    f"grouped split -- the grouped number is the one reported above and the one "
    f"this paper trusts."
)


Model vs baseline, same grouped test split, same label:

method  baseline  logistic_regression  random_forest
k                                                   
50         0.340                0.480          0.480
200        0.350                0.510          0.525
500        0.344                0.542          0.584

Test-set base rate: 0.511  (n_test=6,163)



Split honesty (ML-09): Random Forest precision@500 is 0.830 under a random row-level split vs 0.584 under the grouped split -- the grouped number is the one reported above and the one this paper trusts.


## 5. Limitations

*What this work cannot claim.*

In [5]:
print(
    "What this work cannot claim:\n"
    "  - No causal claim. This is cross-sectional, observational data -- "
    "'declining pages tend to be stale and visible' is an association, not "
    "proof that refreshing a page prevents decline.\n"
    "  - Not a Google-algorithm model. trend_direction reflects THIS portfolio's "
    "impression trend, not search-engine ranking mechanics.\n"
    "  - Client generalization is untested beyond the clients in this dataset -- "
    "the grouped split proves the model works on UNSEEN PAGES from clients like "
    "these, not on a genuinely new client's content style.\n"
    "  - Precision@K (0.48-0.58) means the model is directionally useful for "
    "prioritization, not a reliable per-row verdict -- roughly 4-6 of every 10 "
    "flagged pages are genuinely declining.\n"
    "  - Feature importance (ML-08) is descriptive of what the model leans on, "
    "not proof of what drives real-world decline -- the same caveat the FlyRank "
    "paper itself applies to its own Random Forest appendix (ML-09, Finding A)."
)


What this work cannot claim:
  - No causal claim. This is cross-sectional, observational data -- 'declining pages tend to be stale and visible' is an association, not proof that refreshing a page prevents decline.
  - Not a Google-algorithm model. trend_direction reflects THIS portfolio's impression trend, not search-engine ranking mechanics.
  - Client generalization is untested beyond the clients in this dataset -- the grouped split proves the model works on UNSEEN PAGES from clients like these, not on a genuinely new client's content style.
  - Precision@K (0.48-0.58) means the model is directionally useful for prioritization, not a reliable per-row verdict -- roughly 4-6 of every 10 flagged pages are genuinely declining.
  - Feature importance (ML-08) is descriptive of what the model leans on, not proof of what drives real-world decline -- the same caveat the FlyRank paper itself applies to its own Random Forest appendix (ML-09, Finding A).


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [6]:
# This mirrors ML-10's action_playbook_queue.csv -- the paper's recommendations
# section is this table, not a rebuild of the model.
import os

if os.path.exists('work/outputs/action_playbook_queue.csv'):
    queue = pd.read_csv('work/outputs/action_playbook_queue.csv')
    print("Loaded work/outputs/action_playbook_queue.csv (from ML-10)\n")
else:
    # Fallback: recompute inline if the CSV hasn't been generated in this session yet
    rf_full = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1).fit(X, y)
    df['predicted_decline_probability'] = rf_full.predict_proba(X)[:, 1].round(4)
    q90, q70 = df['predicted_decline_probability'].quantile([0.90, 0.70])
    df['action_label'] = np.select(
        [df['predicted_decline_probability'] >= q90, df['predicted_decline_probability'] >= q70],
        ['refresh_now', 'review_soon'], default='monitor',
    )
    queue = df.sort_values('predicted_decline_probability', ascending=False)

print("Top 10 recommended reviews:")
cols = [c for c in ['content_id', 'action_rank', 'predicted_decline_probability',
                     'action_label', 'reason_code'] if c in queue.columns]
print(queue[cols].head(10).to_string(index=False))
print(f"\nFull queue: {len(queue):,} pages ranked.")


Loaded work/outputs/action_playbook_queue.csv (from ML-10)

Top 10 recommended reviews:
          content_id  action_rank  predicted_decline_probability action_label   reason_code
content_45f96356f84e            1                         0.7954  refresh_now model_flagged
content_123b5d750cbc            2                         0.7921  refresh_now model_flagged
content_0cbf5c93e96e            3                         0.7877  refresh_now model_flagged
content_ff8019ba021d            4                         0.7847  refresh_now model_flagged
content_20c5d6a1c7b1            5                         0.7818  refresh_now model_flagged
content_7b2b7acfcc4a            6                         0.7786  refresh_now model_flagged
content_1e446b05f1c5            7                         0.7776  refresh_now model_flagged
content_b23fa9e12c1d            8                         0.7762  refresh_now model_flagged
content_65bf969d1247            9                         0.7755  refresh_now model_

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [7]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

os.makedirs('docs/img', exist_ok=True)

# Chart 1: model vs baseline precision@K
ks = [50, 200, 500]
baseline_p = [precision_at_k(baseline_test.values, yte.values, k) for k in ks]
lr_p = [precision_at_k(p_lr, yte.values, k) for k in ks]
rf_p = [precision_at_k(p_rf, yte.values, k) for k in ks]

fig, ax = plt.subplots(figsize=(7, 4.5))
width = 0.25
x = np.arange(len(ks))
ax.bar(x - width, baseline_p, width, label='Baseline (Week-4 rule)', color='#94a3b8')
ax.bar(x, lr_p, width, label='Logistic Regression', color='#60a5fa')
ax.bar(x + width, rf_p, width, label='Random Forest', color='#1d4ed8')
ax.axhline(yte.mean(), color='#ef4444', linestyle='--', linewidth=1, label=f'Base rate ({yte.mean():.2f})')
ax.set_xticks(x); ax.set_xticklabels([f'K={k}' for k in ks])
ax.set_ylabel('Precision@K')
ax.set_title('Model vs. baseline: precision@K on the same grouped test split')
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig('docs/img/precision_at_k.png', dpi=150)
plt.close(fig)

# Chart 2: CTR by position tier (ML-06 signal 3)
order = ['top_3', 'page_1', 'striking', 'page_3_5', 'deep']
t = df.groupby('position_tier')['ctr'].mean().reindex(order)
fig2, ax2 = plt.subplots(figsize=(7, 4))
ax2.bar(t.index, t.values, color='#1d4ed8')
ax2.set_ylabel('Average CTR (%)')
ax2.set_title('CTR falls as position tier worsens (ML-06, Signal 3)')
fig2.tight_layout()
fig2.savefig('docs/img/position_ctr.png', dpi=150)
plt.close(fig2)

# Chart 3: split-honesty gap
tr_r2, te_r2 = train_test_split(np.arange(len(X)), test_size=0.2, random_state=42, stratify=y)
rf_r2 = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1).fit(X.iloc[tr_r2], y.iloc[tr_r2])
p_rf_r2 = rf_r2.predict_proba(X.iloc[te_r2])[:, 1]
rf_random_p = [precision_at_k(p_rf_r2, y.iloc[te_r2].values, k) for k in ks]

fig3, ax3 = plt.subplots(figsize=(7, 4))
ax3.plot(ks, rf_random_p, marker='o', label='Random row-level split', color='#f97316')
ax3.plot(ks, rf_p, marker='o', label='Grouped by client_id (honest)', color='#1d4ed8')
ax3.set_xlabel('K'); ax3.set_ylabel('Precision@K')
ax3.set_title('Split design changes the score: random split inflates precision')
ax3.legend(fontsize=8)
fig3.tight_layout()
fig3.savefig('docs/img/split_gap.png', dpi=150)
plt.close(fig3)

print("Saved: docs/img/precision_at_k.png, docs/img/position_ctr.png, docs/img/split_gap.png")
print("These are the exact three charts the deployed paper (docs/index.html) embeds.")


Saved: docs/img/precision_at_k.png, docs/img/position_ctr.png, docs/img/split_gap.png
These are the exact three charts the deployed paper (docs/index.html) embeds.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.